In [1]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from base64 import b64decode
from dotenv import load_dotenv
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if os.getenv(f"GITHUB_TOKEN_{i}")]
token_index = 0

def get_headers():
    global token_index
    headers = {"Authorization": f"token {tokens[token_index]}"}
    token_index = (token_index + 1) % len(tokens)
    return headers

# === Paths ===
OUTPUT_DIR = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

step1_output = os.path.join(OUTPUT_DIR, "step1_url_lookup_output.csv")
interim_output = os.path.join(OUTPUT_DIR, "step2_manifest_check_output.csv")
final_output = os.path.join(OUTPUT_DIR, "step2_manifest_final_output.csv")

# === Load data with resume logic ===
try:
    df = pd.read_csv(interim_output)
    print("📄 Resuming from interim output...")
except FileNotFoundError:
    df = pd.read_csv(step1_output)
    print("📥 Loaded from Step 1 output.")
    if "valid_repo" in df.columns:
        df = df.rename(columns={"valid_repo": "Valid_Repo_Step1"})
    # Initialize Step 2 columns
    df["has_manifest"] = "no"
    df["has_activity"] = "no"
    df["nbr_of_manifest"] = -1
    df["Standard_Manifest"] = "none"

# === Filter repos to process ===
target_df = df[(df["Valid_Repo_Step1"] == "yes") & (df["nbr_of_manifest"] == -1)].reset_index(drop=True)

# === Process each repo ===
for idx, row in target_df.iterrows():
    html_url = row["html_url"]
    repo_path = html_url.replace("https://github.com/", "")
    print(f"[{idx + 1}/{len(target_df)}] Checking: {html_url}")

    # Get all files in repo
    tree_url = f"https://api.github.com/repos/{repo_path}/git/trees/HEAD?recursive=1"
    r = requests.get(tree_url, headers=get_headers())
    if r.status_code != 200:
        print(f"❌ Failed to fetch tree for {html_url}")
        continue

    paths = [f['path'] for f in r.json().get("tree", []) if "AndroidManifest.xml" in f['path']]
    df.loc[df["html_url"] == html_url, "nbr_of_manifest"] = len(paths)

    if not paths:
        continue

    df.loc[df["html_url"] == html_url, "has_manifest"] = "yes"

    standard_detected = any(
        p == "app/src/main/AndroidManifest.xml" or p.endswith("/src/main/AndroidManifest.xml")
        for p in paths
    )

    for path in paths:
        file_url = f"https://api.github.com/repos/{repo_path}/contents/{path}"
        fr = requests.get(file_url, headers=get_headers())
        if fr.status_code != 200:
            continue
        try:
            content = b64decode(fr.json().get("content", "")).decode("utf-8", errors="ignore")
            if "<activity" in content:
                df.loc[df["html_url"] == html_url, "has_activity"] = "yes"
                break
        except Exception:
            continue

    df.loc[df["html_url"] == html_url, "Standard_Manifest"] = "yes" if standard_detected else "no"

    # ✅ Save interim progress after each repo
    df.to_csv(interim_output, index=False)

# === Finalize Step 2 Validity Flag ===
df["Valid_Repo_Step2"] = df.apply(lambda r: "yes" if r["has_activity"] == "yes" else "no", axis=1)

# === Save final output ===
df.to_csv(final_output, index=False)
print(f"✅ Step 2 complete. Output saved to:\n{final_output}")


📥 Loaded from Step 1 output.
[1/75053] Checking: https://github.com/dustin/java-memcached-client
[2/75053] Checking: https://github.com/mth/yeti
[3/75053] Checking: https://github.com/cyberfox/jbidwatcher
[4/75053] Checking: https://github.com/mheath/adbcj
[5/75053] Checking: https://github.com/jashkenas/ruby-processing
[6/75053] Checking: https://github.com/davidB/scala-maven-plugin
[7/75053] Checking: https://github.com/rictic/code_swarm
[8/75053] Checking: https://github.com/rikrd/geomerative
[9/75053] Checking: https://github.com/slagyr/limelight
[10/75053] Checking: https://github.com/leachim6/hello-world
[11/75053] Checking: https://github.com/brendanlim/mobile-fu
[12/75053] Checking: https://github.com/clarkware/jdepend
[13/75053] Checking: https://github.com/mhroth/jvsthost
[14/75053] Checking: https://github.com/bmc/javautil
[15/75053] Checking: https://github.com/darkk/redsocks
[16/75053] Checking: https://github.com/sintaxi/phonegap
[17/75053] Checking: https://github.com/rh